# TIW-DSM reproduction: run everything on Kaggle / Colab

**Kaggle settings:** Accelerator = GPU (T4 x1 or P100), Internet = ON, Persistence = Files.
For long runs use **Save Version, then Save & Run All (Commit)**. It runs in the background for up to 12 h; you do not need to keep the browser open.

Split the work across team accounts, then collect the outputs:
| Who | `TASK` | Time on T4 (approx.) |
|---|---|---|
| Member 1 | `"disc"`, then `"obs"` | 1 h + about 10 h |
| Member 2 | `"tiw"` (needs `discriminator.pt` from member 1) | about 11 h |
| Member 3 | `"eval"` after both finish (+ the app screenshots) | 2-3 h |

To resume or share outputs: add the previous version's output as an **Input** (Add data, then Notebook output), then set `PREV` below.

In [ ]:
TASK = "bench"   # one of: bench | disc | obs | tiw | eval | toy
REPO = "https://github.com/<YOUR_GITHUB>/tiw-dsm-repro.git"
PREV = ""        # e.g. "/kaggle/input/tiw-dsm-run-tiw" : folder with earlier runs/ or checkpoints/ to resume from
DURATION_MIMG = 11   # same value for obs and tiw (set from the benchmark cell)
BATCH = 256

In [ ]:
import os, subprocess, shutil
if not os.path.exists("tiw-dsm-repro"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("tiw-dsm-repro")
!pip -q install -r requirements.txt
if PREV:
    for sub in ["runs", "checkpoints", "results", "stats"]:
        src = os.path.join(PREV, "tiw-dsm-repro", sub) if os.path.exists(os.path.join(PREV, "tiw-dsm-repro")) else os.path.join(PREV, sub)
        if os.path.exists(src):
            shutil.copytree(src, sub, dirs_exist_ok=True); print("restored", sub)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python scripts/prepare_data.py
!cat results/data_info.json

### Benchmark (about 10 min): measure the speed, then choose `DURATION_MIMG` so that TIW-DSM fits in about 11 h.
It needs `checkpoints/discriminator.pt`. If you don't have it yet, the cell trains a 300-iteration throwaway discriminator for timing only.

In [ ]:
if TASK == "bench":
    if not os.path.exists("checkpoints/discriminator.pt"):
        !python scripts/train_discriminator.py --iters 300 --out checkpoints/bench_disc.pt
        disc = "checkpoints/bench_disc.pt"
    else:
        disc = "checkpoints/discriminator.pt"
    for mode in ["obs", "tiw"]:
        !python scripts/train_score.py --mode {mode} --outdir runs/bench_{mode} --duration_mimg 0.1 --batch {BATCH} --disc {disc} --snapshot_kimg 1000
    import json
    for mode in ["obs", "tiw"]:
        rec = [json.loads(l) for l in open(f"runs/bench_{mode}/log.jsonl")][-1]
        s = rec["sec_per_kimg"]; print(f"{mode}: {s:.2f} s/kimg  ->  {11*3600/s/1000:.1f} Mimg fit in 11 h")
    print("Use the TIW value (the slower one) for DURATION_MIMG, capped at 11 (the paper's budget).")

In [ ]:
if TASK == "disc":
    !python scripts/train_discriminator.py --iters 10000
    !cat results/discriminator_diag.json

In [ ]:
if TASK in ("obs", "tiw"):
    !python scripts/train_score.py --mode {TASK} --outdir runs/{TASK} --duration_mimg {DURATION_MIMG} --batch {BATCH} --time_limit_h 11.3
    # If the time limit hits first: commit again with PREV pointing at this output; training resumes automatically.

In [ ]:
if TASK == "toy":
    !python scripts/toy_2d.py

In [ ]:
if TASK == "eval":
    !python scripts/evaluate.py --ckpt runs/obs/ema_final.pt --name obs --num 50000
    !python scripts/evaluate.py --ckpt runs/tiw/ema_final.pt --name tiw --num 50000
    for r in ["bias", "ref", "target"]:
        !python scripts/evaluate.py --real {r} --name data_{r}
    !python scripts/steps_experiment.py --ckpt obs=runs/obs/ema_final.pt --ckpt tiw=runs/tiw/ema_final.pt --num 10000
    !python scripts/make_report_assets.py --runs obs=runs/obs tiw=runs/tiw

### Collect for the report and the app
Download `results/`, `report/figures/`, `report/generated/`, `runs/*/log.jsonl`, `runs/*/samples_*.png` and `runs/*/ema_final.pt`.
Copy `runs/tiw/ema_final.pt` to `checkpoints/deploy/tiw_dsm.pt` and `runs/obs/ema_final.pt` to `checkpoints/deploy/dsm_obs.pt` for the Streamlit app.

In [ ]:
!mkdir -p /kaggle/working/export && cp -r results report/figures report/generated /kaggle/working/export/ 2>/dev/null; ls /kaggle/working/export